# SQL Server → Databricks Unity Catalog | One-Time JDBC Load

Reads data from a SQL Server database via JDBC and writes it to a Unity Catalog bronze table.  
Safe to rerun — uses `CREATE OR REPLACE TABLE` semantics on every execution.

In [ ]:
# ------------------------------------------------------------
# 1. Configuration — fill in all values before running
# ------------------------------------------------------------

# JDBC connection settings
jdbc_hostname = "your-sql-server.database.windows.net"
jdbc_port     = 1433
jdbc_database = "your_database_name"
jdbc_user     = dbutils.secrets.get(scope="your-scope", key="sql-server-user")
jdbc_password = dbutils.secrets.get(scope="your-scope", key="sql-server-password")

# Unity Catalog target
uc_catalog = "your_catalog"
uc_schema  = "bronze"

# SQL query to run against the source — edit this
# The target table name is derived from the table referenced in FROM
sql_query = "select * from schema_name.table_name"

In [ ]:
# ------------------------------------------------------------
# 2. Derive the target table name from the query
#    Grabs the last segment after the final dot in FROM clause
# ------------------------------------------------------------
import re

match = re.search(r"from\s+[\w.]*?(\w+)\s*$", sql_query, re.IGNORECASE)
if not match:
    raise ValueError(f"Could not parse a table name from query: {sql_query}")

target_table_name = match.group(1).lower()
full_target_table = f"{uc_catalog}.{uc_schema}.{target_table_name}"
print(f"Target table: {full_target_table}")

In [ ]:
# ------------------------------------------------------------
# 3. Build JDBC URL and connection properties
# ------------------------------------------------------------

jdbc_url = (
    f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};"
    f"database={jdbc_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user":     jdbc_user,
    "password": jdbc_password,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

In [ ]:
# ------------------------------------------------------------
# 4. Read from SQL Server
# ------------------------------------------------------------

df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table=f"({sql_query}) AS src",
        properties=connection_properties,
    )
)

print(f"Rows read: {df.count()}")
df.printSchema()

In [ ]:
# ------------------------------------------------------------
# 5. Write to Unity Catalog — CREATE OR REPLACE TABLE semantics
#    Safe to rerun: replaces data and schema on every execution
# ------------------------------------------------------------

(
    df.writeTo(full_target_table)
    .using("delta")
    .createOrReplace()
)

print(f"Done. Table created or replaced: {full_target_table}")